## Pre-trained GPT 모델 Fine-tuning 하기
- fine-tuning에 필요한 데이터셋은 Chat API와 동일한 형식의 **대화**형태여야하며, 각 메시지에 role별로 content가 들어간 메시지 목록이어야 함(system, user, assistant)
- OpenAI의 fine-tuning API는 최소 10개이상의 대화 데이터가 필요하며, 비용상 처음부터 많은 양으로 진행하기보단 50개 미만의 소량 데이터셋으로 학습시켜 결과 먼저 확인 -> 필요시 추가
- OpenAI공식 페잉지에서는 100개 미만의 데이터를 소량의 데이터로 정의하고 있음
- fine-tuning전에 모델에 가장 잘 맞는 지침 프롬프트를 설계한 뒤 해당 텍스트가 포함된 학습 데이터를 제작하는 것이 좋음
- 일부 학습 데이터에는 기존 모델이 **원하는대로 작동하지 않았을 경우** 제대로 된 응답 데이터를 포함시켜야 성능 향상
    - **원하는대로 작동하지 않았던 응답** Q. 프랑스 수도 어디야? A. 프랑스는 세계에서 가장 유명한 관광도시예요!
    - **fine-tuning에 넣어줄 제대로 된 데이터** Q. 프랑스 수도 어디야? A. 프랑스 수도는 파리입니다.

In [2]:
# openAI 모델들이 사용하는 base토큰화 및 인코딩 지원 모듈
!pip install tiktoken

In [3]:
import os
import time
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime, UTC
import tiktoken
# 딕셔너리에 존재하지 않는 key에 대한 접근시, 자료형에 따라 디폴트 값을 생성해주는 클래스
from collections import defaultdict

In [6]:
load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=MY_API_KEY)

### 1. Fine-tuning 전 데이터 로드 및 검증 단계
- 비용이 많이 들 수 있기 때문에 사전에 적합한 데이터 형태인지를 먼저 검증하고 진행하기
- OpenAI의 fine-tuning API 사용시 데이터 구조는 JSONL이어야 함
- JSONL(JSON Lines): 일반 JSON이 전체 데이터를 하나의 객체로 보는 것에 비해, JSONL은 한줄당 JSON객체가 존재하는 형태. 한 줄씩 또는 청크 단위로 읽고 쓰는 것에 효율적. 대규모 데이터 세트를 구성하는데 적합

#### 1) 데이터를 직접 JSONL 파일로 저장하기

In [ ]:
data = [
    {"messages": [
        {"role":"system", "content": "You are a chatbot that gives clear answers."},
        {"role":"user", "content":"What can i do?"},
        {"role":"assistant", "conetent": "Ask a question about the part you want."}
    ]},
    {"messages": [
        {"role":"system", "content": "You are a chatbot that gives clear answers."},
        {"role":"user", "content":"What can i do?"},
        {"role":"assistant", "conetent": "Ask a question about the part you want."}
    ]}
]

with open('data/my_dataset.jsonl', 'w', encoding='utf-8') as f:
    # json파일 저장시 한 줄에 json객체(딕셔너리)를 하나씩 저장해주는 방식
    for i in data :
        f.write(json.dumps(i) + '\n')